# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Statistical Thinking for Model Evaluation
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from statsmodels.stats.contingency_tables import mcnemar

np.random.seed(42)

# Binary classification dataset: credit risk prediction
# 20 features: model_a uses all 20, model_b uses only the first 10
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    weights=[0.7, 0.3],
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(20)]
df = pd.DataFrame(X, columns=feature_names)
df["default"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# model_a: all 20 features
model_a = LogisticRegression(max_iter=1000, random_state=42)
model_a.fit(X_train, y_train)

# model_b: first 10 features only
model_b = LogisticRegression(max_iter=1000, random_state=42)
model_b.fit(X_train[:, :10], y_train)

proba_a = model_a.predict_proba(X_test)[:, 1]
proba_b = model_b.predict_proba(X_test[:, :10])[:, 1]

pred_a = model_a.predict(X_test)
pred_b = model_b.predict(X_test[:, :10])

print(f"Test set size: {len(y_test)}")
print(f"Class balance — 0: {(y_test == 0).sum()}, 1: {(y_test == 1).sum()}")
print(f"Model A AUC (point estimate): {roc_auc_score(y_test, proba_a):.4f}")
print(f"Model B AUC (point estimate): {roc_auc_score(y_test, proba_b):.4f}")

## Part 1: Bootstrap Confidence Intervals

In [ ]:
def bootstrap_auc(y_true, y_score, n_iterations=1000, ci=0.95):
    """Bootstrap 95% CI for AUC using stratified resampling of the test set."""
    aucs = []
    rng = np.random.RandomState(0)
    alpha = (1 - ci) / 2

    for _ in range(n_iterations):
        indices = resample(np.arange(len(y_true)), replace=True, random_state=rng)
        y_boot = y_true[indices]
        s_boot = y_score[indices]

        # Guard against degenerate bootstrap samples (only one class present)
        if len(np.unique(y_boot)) < 2:
            continue

        aucs.append(roc_auc_score(y_boot, s_boot))

    aucs = np.array(aucs)
    lower = np.percentile(aucs, 100 * alpha)
    upper = np.percentile(aucs, 100 * (1 - alpha))
    point = roc_auc_score(y_true, y_score)
    return point, lower, upper


auc_a, lo_a, hi_a = bootstrap_auc(y_test, proba_a)
auc_b, lo_b, hi_b = bootstrap_auc(y_test, proba_b)

print("Bootstrap AUC with 95% Confidence Intervals")
print("=" * 48)
print(f"Model A (all 20 features): AUC = {auc_a:.4f}  CI = [{lo_a:.4f}, {hi_a:.4f}]")
print(f"Model B (10 features):     AUC = {auc_b:.4f}  CI = [{lo_b:.4f}, {hi_b:.4f}]")
print()
print(f"CI overlap: {lo_a:.4f}-{hi_a:.4f}  vs  {lo_b:.4f}-{hi_b:.4f}")

**Question 1:** The two models differ by roughly 2 AUC points, but their confidence intervals overlap substantially. Can you conclude that model_a is better than model_b? What would you need to see in the confidence intervals to make a confident claim of improvement?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Overlapping CIs are not conclusive:** Overlapping confidence intervals mean the observed difference is consistent with sampling noise at the 95% confidence level — you cannot conclude model_a is better. CI overlap does not equal "no difference," but it does mean you lack sufficient evidence to assert one.

**What you need:** The lower bound of model_a's CI must exceed the upper bound of model_b's CI (non-overlapping intervals) to make a confident directional claim. Alternatively, apply McNemar's test directly on the paired predictions for a formal paired significance test, which is more powerful than comparing CIs visually.

</details>

**Question 2:** The `bootstrap_auc` function resamples with replacement from the test set, and it skips any bootstrap sample that contains only one class. When would such degenerate samples occur frequently? How does the size of the test set affect the width of the resulting confidence interval?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**When degenerate samples occur frequently:** When the test set is small and the minority class is rare. If there are only 10 positive examples in the test set, a random bootstrap resample of the same size has a non-trivial probability of drawing none of them — AUC is undefined without both classes present. With a 30% positive class and n=500, degenerate samples are extremely rare; with a 2% class and n=100, they become common.

**Effect of test set size on CI width:** Smaller test sets produce wider confidence intervals. Each bootstrap resample is a noisier estimate of the true AUC because it is drawn from fewer observations; the percentile-based CI reflects this variability directly. More test examples → tighter bootstrap distribution → narrower CI.

</details>

## Part 2: McNemar's Test

In [ ]:
# Indicator arrays: 1 = correct, 0 = incorrect
correct_a = (pred_a == y_test).astype(int)
correct_b = (pred_b == y_test).astype(int)

# McNemar contingency table
# b = A correct, B wrong; c = A wrong, B correct
b = np.sum((correct_a == 1) & (correct_b == 0))  # A wins
c = np.sum((correct_a == 0) & (correct_b == 1))  # B wins

print(f"Discordant pairs — A correct / B wrong (b): {b}")
print(f"Discordant pairs — A wrong / B correct (c): {c}")
print(f"Both correct: {np.sum((correct_a == 1) & (correct_b == 1))}")
print(f"Both wrong:   {np.sum((correct_a == 0) & (correct_b == 0))}")
print()

# Build 2x2 table expected by statsmodels
# [[both_correct, a_correct_b_wrong], [a_wrong_b_correct, both_wrong]]
both_correct = int(np.sum((correct_a == 1) & (correct_b == 1)))
both_wrong = int(np.sum((correct_a == 0) & (correct_b == 0)))
table = np.array([[both_correct, int(b)], [int(c), both_wrong]])

result = mcnemar(table, exact=True)
print(f"McNemar's test (exact): statistic = {result.statistic:.4f}, p-value = {result.pvalue:.4f}")
print()
if result.pvalue < 0.05:
    print("Interpretation: The two models make significantly different errors (p < 0.05).")
else:
    print("Interpretation: No significant difference in the error patterns of the two models (p >= 0.05).")

**Question 3:** Suppose the p-value were 0.21. What would that mean about the two classifiers? Why is McNemar's test more appropriate here than a t-test on the difference in accuracy between the two models?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**p = 0.21 interpretation:** A p-value of 0.21 means the observed asymmetry in discordant pairs (cases where one model is correct and the other is wrong) is consistent with chance variation — there is no statistically significant difference in the two models' error patterns at the 5% level.

**Why McNemar's, not a t-test:** A t-test on accuracy assumes independent measurements, but both models are evaluated on exactly the same test examples. Their errors are correlated. McNemar's test is designed for paired binary outcomes: it only uses the discordant pairs (where the models disagree) and tests whether one model is systematically better at the cases the other gets wrong. The concordant pairs carry no information about which model is better — they cancel out.

</details>

**Question 4:** McNemar's test requires both models to be evaluated on the *exact same* test set. Suppose model_b had been tuned (e.g., threshold adjusted, features selected) after inspecting the test set predictions. What would this do to the validity of the McNemar result, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**The validity is destroyed.** McNemar's test assumes that both models' predictions on each test example are determined independently of the test labels, except through the training process. If model_b was tuned (threshold adjusted, features selected) *after* inspecting test-set predictions, those predictions are no longer independent of the test labels — model_b has effectively been trained on the test data.

**Consequence:** The discordant pairs now reflect not just model differences but also the deliberate alignment of model_b's predictions with the test set. The p-value will be anticonservatively small, making a difference appear significant when it is a result of overfitting to the test set, not genuine generalisation improvement.

</details>

## Part 3: Learning Curves

In [ ]:
# Learning curves for model_a (all features) using 5-fold CV
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000, random_state=42),
    X, y,
    train_sizes=np.linspace(0.1, 1.0, 9),
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, "o-", color="steelblue", label="Training AUC")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="steelblue")
ax.plot(train_sizes, val_mean, "o-", color="darkorange", label="Validation AUC")
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="darkorange")
ax.set_xlabel("Training set size")
ax.set_ylabel("ROC AUC")
ax.set_title("Learning Curve — Logistic Regression (all 20 features)")
ax.legend(loc="lower right")
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

print(f"Final training AUC:   {train_mean[-1]:.4f} (+/- {train_std[-1]:.4f})")
print(f"Final validation AUC: {val_mean[-1]:.4f} (+/- {val_std[-1]:.4f})")
print(f"Gap (train - val):    {train_mean[-1] - val_mean[-1]:.4f}")

**Question 5:** Suppose the training and validation curves converged at an AUC of 0.70, well below what you believe the problem warrants. What does this pattern diagnose — high bias or high variance? What concrete steps would you try next?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Diagnosis: high bias (underfitting).** When both the training and validation curves converge and plateau at a performance level well below what the problem warrants, the model cannot fit the signal in the data regardless of how much data you add — this is a bias problem, not a variance problem.

**What to try next:**
- Switch to a more expressive model class (gradient boosting, neural network, or a random forest instead of logistic regression)
- Engineer interaction or polynomial features to give the model more signal to work with
- Reduce regularization (e.g., increase `C` in logistic regression)
- Investigate whether important predictive features are missing from the dataset entirely

Adding more data will not help — the curves have already converged.

</details>

**Question 6:** Suppose the validation curve is still rising at the maximum training size in this plot — it has not yet flattened. What does this tell you about what would help the model? What would *not* help? Consider: more data, more regularisation, more features, a more complex model.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**What it tells you:** Validation still rising at maximum training size means the model is data-limited — it can still improve with more training examples. The bottleneck is data volume, not model capacity.

**What helps:** More training data is the right lever — it would continue the upward trend of the validation curve.

**What would NOT help:**
- More regularisation: would reduce model capacity and lower both curves
- A simpler model: same problem — reduces capacity, doesn't address the data shortage
- More features: might help if the features are informative, but the rising curve alone doesn't diagnose missing features — it diagnoses insufficient data for the current feature set
- Increasing complexity (deeper trees, more layers): the model is already able to learn; the issue is lack of data to generalise

</details>

## Summary

Before moving on, check your understanding with these final questions:

1. A bootstrap CI for AUC that reads [0.71, 0.79] means you are 95% confident the true AUC lies in that interval — what does "true AUC" refer to, and why can you never measure it directly?

2. McNemar's test operates on the *discordant pairs* between two classifiers — pairs where one is correct and the other is wrong. Why do the concordant pairs (both right or both wrong) carry no information for this test?

3. Learning curves are plotted using cross-validation, not a held-out test set. What would happen to your test-set budget if you used the test set to generate every point on a learning curve?

*(Write your answers here.)*

<details>
<summary>🔑 Reveal summary answers</summary>

1. **True AUC:** The population AUC over the entire data-generating process — the theoretical performance you would get if you could evaluate on an infinite number of examples drawn from the same distribution. You can never measure it directly because your test set is a finite sample; the "true" value is a property of the population, not any fixed dataset.

2. **Concordant pairs carry no information:** McNemar's test asks only "when the models disagree, does one win more than the other?" Pairs where both models are right or both are wrong tell us nothing about relative performance — they cancel out of the statistic entirely. Only the asymmetry in disagreements (b vs c) contains evidence about which model generalises better.

3. **Test-set budget:** Each learning curve point would "spend" some of the test set's labels to estimate performance at that training size. Repeated evaluation on the same test set inflates the apparent performance because you are effectively searching for the training size that gives the best result — making the final held-out estimate optimistic and the test set no longer truly held out.

</details>